# T5-large training with LoRA


In [ ]:
# Install once if needed:
# %pip install -U transformers datasets peft accelerate sentencepiece

## 1. Load the original datasets

Run this only once to create the Splits

In [ ]:
from datasets import load_dataset, DatasetDict

# Load the full dataset
data = load_dataset(
    "json",
    data_files="full_dataset/full_dataset.json"
)["train"]

# Split training data
split = data.train_test_split(train_size=1022, seed=42)

# Split remaining data into validation and test
remaining = split["test"].train_test_split(
    train_size=137,
    seed=42
)

# Create DatasetDict
dataset = DatasetDict({
    "train": split["train"],
    "val": remaining["train"],
    "test": remaining["test"]
})



dataset["train"].to_json("datasplits/train.json")
dataset["val"].to_json("datasplits/val.json")
dataset["test"].to_json("datasplits/test.json")

Run this every time to get the splits

In [5]:
from datasets import load_dataset

ds = load_dataset("json", data_files={
    "train": "datasplits/train.json",
    "val": "datasplits/val.json",
    "test": "datasplits/test.json"
})
print(ds)
print(ds["train"][0])

DatasetDict({
    train: Dataset({
        features: ['question', 'answer', 'context'],
        num_rows: 1022
    })
    val: Dataset({
        features: ['question', 'answer', 'context'],
        num_rows: 137
    })
    test: Dataset({
        features: ['question', 'answer', 'context'],
        num_rows: 117
    })
})
{'question': 'Do learning agreements define recognition before mobility?', 'answer': 'Yes. They set the terms for recognition. Recognition follows the signed learning agreement and ECTS principles; equivalence is based on learning outcomes, content similarity, and workload.', 'context': 'Mobility and recognition (e.g., ERASMUS) follow ECTS principles and a pre‑approved learning agreement signed by the student, the home department, and the host institution. After mobility, the host transcript is matched to the agreement, and the Department considers content, learning outcomes, and workload to determine equivalence and transfer of credits. Recognition may be full or par

## 2. Prepare question/context inputs and answers

In [6]:
def process(dataset):
    input_data = [f"question: {q} context: {c}" for q, c in zip(dataset["question"], dataset["context"])]
    return {"input": input_data, "target": dataset["answer"]}

processed_data = ds.map(process, batched=True, batch_size=100,
                        remove_columns=ds["train"].column_names)
print(processed_data["train"][0])

{'input': 'question: Do learning agreements define recognition before mobility? context: Mobility and recognition (e.g., ERASMUS) follow ECTS principles and a pre‑approved learning agreement signed by the student, the home department, and the host institution. After mobility, the host transcript is matched to the agreement, and the Department considers content, learning outcomes, and workload to determine equivalence and transfer of credits. Recognition may be full or partial depending on the match; grade conversions are applied where required by institutional policy. Deadlines and application steps are published by International Relations and the Department.', 'target': 'Yes. They set the terms for recognition. Recognition follows the signed learning agreement and ECTS principles; equivalence is based on learning outcomes, content similarity, and workload.'}


## 3. Tokenize for T5-large


In [ ]:
from transformers import T5Tokenizer

tokenizer = T5Tokenizer.from_pretrained("t5-large")

print("Tokenizer loaded successfully")

In [7]:



def tokenizator(examples):
    inputs = tokenizer(
        examples["input"],
        max_length=512,
        truncation=True,
        padding="max_length"
    )

    labels = tokenizer(
        examples["target"],
        max_length=128,
        truncation=True,
        padding="max_length"
    )["input_ids"]

    inputs["labels"] = [
        [-100 if token == tokenizer.pad_token_id else token for token in label]
        for label in labels
    ]

    return inputs

tokenized_data = processed_data.map(
    tokenizator,
    batched=True,
    batch_size=16
)

Map:   0%|          | 0/1022 [00:00<?, ? examples/s]

Map:   0%|          | 0/137 [00:00<?, ? examples/s]

Map:   0%|          | 0/117 [00:00<?, ? examples/s]

In [10]:
print(tokenized_data)

DatasetDict({
    train: Dataset({
        features: ['input', 'target', 'input_ids', 'attention_mask', 'labels'],
        num_rows: 1022
    })
    val: Dataset({
        features: ['input', 'target', 'input_ids', 'attention_mask', 'labels'],
        num_rows: 137
    })
    test: Dataset({
        features: ['input', 'target', 'input_ids', 'attention_mask', 'labels'],
        num_rows: 117
    })
})


## 4. Load T5-large and add LoRA


In [8]:
import torch
from transformers import T5ForConditionalGeneration
from peft import LoraConfig, get_peft_model, TaskType

model = T5ForConditionalGeneration.from_pretrained(
    "t5-large",
    torch_dtype=torch.bfloat16
)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=["q", "k", "v", "o", "wi_0", "wi_1", "wo"],
    task_type=TaskType.SEQ_2_SEQ_LM
)

model = get_peft_model(model, lora_config)
model.config.use_cache = False

model.print_trainable_parameters()

trainable params: 13,369,344 || all params: 751,037,440 || trainable%: 1.7801


## 5. Train


In [11]:
from transformers import Trainer, TrainingArguments

training_args = TrainingArguments(
    output_dir="./t5_large_lora_training",
    num_train_epochs=3,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=16,
    learning_rate=5e-5,
    bf16=True,
    fp16=False,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_steps=10,
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_data["train"],
    eval_dataset=tokenized_data["val"]
)

trainer.train()

Epoch,Training Loss,Validation Loss
1,2.098300,1.935526
2,1.626000,1.637851
3,1.752200,1.603405


TrainOutput(global_step=192, training_loss=2.041646429648002, metrics={'train_runtime': 2408.0111, 'train_samples_per_second': 1.273, 'train_steps_per_second': 0.08, 'total_flos': 6763957545074688.0, 'train_loss': 2.041646429648002, 'epoch': 3.0})

## 6. Save the new LoRA adapter

In [ ]:
model.save_pretrained("./t5-large-lora-final")
tokenizer.save_pretrained("./t5-large-lora-final")
print("Saved new adapter and tokenizer to ./t5-large-lora-final")

## Evaluation

In [13]:
model.eval()

predictions = []
references = []

for example in processed_data["test"]:
    inputs = tokenizer(
        example["input"],
        return_tensors="pt",
        max_length=512,
        truncation=True
    ).to("cuda")

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=128,
            num_beams=4
        )

    predictions.append(
        tokenizer.decode(outputs[0], skip_special_tokens=True)
    )
    references.append(example["target"])

In [15]:
import evaluate

rouge = evaluate.load("rouge")

results = rouge.compute(
    predictions=predictions,
    references=references,
    rouge_types=["rouge1", "rouge2", "rougeL"]
)

for metric, score in results.items():
    print(f"{metric}: {score:.4f}")

rouge1: 0.4570
rouge2: 0.2932
rougeL: 0.4098
